# 04: Experimentos Controlados: Dados Brutos × Framework

Etapas 2, 4 e 5 do procedimento metodológico (Seção 3.4): aplicar o pipeline de normalização e balanceamento e
comparar a Rede Bayesiana treinada com **dados brutos** e com **dados tratados pelo framework**.

> **H1:** Espera-se que o pré-processamento estruturado aumente o Recall e a PR-AUC do modelo sobre a classe de ausência.
>
> **H3:** Espera-se que o modelo treinado com a matriz processada pelo pipeline supere, em PR-AUC e ROC-AUC sob validação cruzada, o mesmo modelo treinado com dados brutos.

## Desenho experimental

**Ablação em degraus.** Cada configuração acrescenta um componente do framework, para isolar o efeito de cada um:

| Config. | Dados | Escalonador | SMOTENC |
|---|---|---|---|
| **C0 bruto** (controle) | colunas originais, sem limpeza nem engenharia | - | - |
| C0 bruto + SMOTENC | colunas originais | Robust | ✓ |
| **C1 engenharia** | variáveis do notebook 02 (limpeza, histórico, clima corrigido) | - | - |
| C2 engenharia + escalonador | idem | MinMax / Standard / Robust | - |
| **C3 engenharia + SMOTENC** (framework completo) | idem | nenhum / MinMax / Standard / Robust | ✓ |

**Por que o controle não é "100% bruto".** Uma Rede Bayesiana discreta não aceita números contínuos, então todas as
configurações passam pela mesma discretização por quantis. O controle C0 é a base como chega: `age` errada, clima
com o artefato de "frio intenso", sem histórico e sem ID. Só saem a coluna de vazamento (`no_show_reason`) e as datas
cruas.

**Em todas as configurações e folds:**
1. o escalonador, o SMOTENC e a discretização são ajustados **só no treino**; o teste nunca é reamostrado;
2. a estrutura é aprendida no treino (Hill Climbing com BIC no protocolo principal revisado) e reduzida ao Manto de
   Markov;
3. as CPTs são estimadas no treino (priori BDeu).

**Duas validações:**
- **Validação cruzada** 5 folds × 3 repetições, agrupada por paciente e por registros originais indistinguíveis (15 pares para o Wilcoxon). É o desenho da H3.
- **Validação temporal:** treino até 2020 e teste em 2021–2022. É uma avaliação retrospectiva fora do período de treino, com histórico atualizado sequencialmente e clima observado. Ano e período são retirados dos preditores nesse cenário. A disponibilidade conjunta dos campos antes da consulta não foi validada; o uso antecipado em lembretes permanece como trabalho futuro.

O cálculo completo é feito por `scripts/run_h3.py`. Este notebook carrega os resultados de
`resultados/artigo_sem_sobreposicao/completo/` (mude `RECALCULAR = True` para refazer tudo).

**Protocolo ajustado do artigo.** Mantêm-se os sete contrastes e a RB com BIC como critério principal. No treino interno, escolhem-se três ou cinco faixas e ESS em {1, 10, 100}. Com SMOTENC, escolhe-se também a razão minoritária/majoritária em {0,25; 0,50; 1,00}, mantendo k=5. O critério é Average Precision; empates seguem a ordem predefinida da grade. Controle e engenharia têm a mesma oportunidade de ajuste de faixas. Categorias e cidades são aprendidas no treino. A taxa histórica dos sintéticos é recalculada das contagens e não participa da escolha de vizinhos como atributo redundante. As correções de idade/clima e da proporção de classes permanecem. A avaliação externa inclui recall esperado com orçamento de 10% dos agendamentos e desempate uniforme. Os folds repetidos não são independentes e o teste temporal já foi explorado no desenvolvimento; as conclusões são exploratórias.


**Correção das divisões.** Pacientes identificados e registros com os mesmos campos originais ficam em um único grupo de validação, inclusive sem ID. A assinatura inclui data e horário e exclui `no_show` e `no_show_reason`. A união dos grupos é transitiva; nenhuma linha é removida e esse identificador não entra no modelo. A regra vale também para a seleção interna dos parâmetros. Uma rodada anterior continha sobreposição de linhas e foi descartada; os resultados atuais ficam em `artigo_sem_sobreposicao/`. Isso não resolve a identificação de pessoas sem cadastro em dias diferentes.


In [ ]:
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display
from scipy import stats

from noshow.data import TARGET
from noshow.evaluation import paired_wilcoxon, patient_groups, repeated_group_folds
from noshow.experiments import fit_predict
from noshow.features import CATEGORICAL, CONTINUOUS
from noshow.pipeline import SCALERS, raw_matrix, smotenc_resample
from noshow.data import RAW_PATH, load_raw
import json

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RECALCULAR = False

if RECALCULAR:
    subprocess.run([sys.executable, str(RAIZ / "scripts" / "run_h3.py")], check=True,
                   env={**os.environ, "PYTHONHASHSEED": "0"})
SAIDA = RAIZ / "resultados" / "artigo_sem_sobreposicao" / "completo"
protocolo_path = SAIDA / "protocolo.json"
if not protocolo_path.exists():
    raise FileNotFoundError("Execute PYTHONHASHSEED=0 .venv/bin/python scripts/run_h3.py antes de analisar o protocolo revisado.")
protocolo = json.loads(protocolo_path.read_text())
assert all(protocolo["stages"].get(s, {}).get("complete") for s in ("cv", "temporal")), "Conclua as duas etapas; piloto não valida hipóteses."
assert protocolo["settings"].get("grouping") == "componentes de paciente reconstruído e campos originais iguais, sem alvo/motivo", "Use os resultados com proteção contra registros idênticos."
CV = pd.read_csv(SAIDA / protocolo["stages"]["cv"]["file"])
TEMP = pd.read_csv(SAIDA / protocolo["stages"]["temporal"]["file"])
print(f"Validação cruzada: {CV.config.nunique()} configurações × {CV.groupby(['rep', 'fold']).ngroups} folds | "
      f"validação temporal: {len(TEMP)} configurações")

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
PCT = mtick.PercentFormatter(1, decimals=0)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
ORDEM = ["C0 bruto", "C0 bruto + SMOTENC (robust)", "C1 engenharia",
         "C3 engenharia + SMOTENC (sem escalonador)", "C3 engenharia + SMOTENC (minmax)",
         "C3 engenharia + SMOTENC (standard)", "C3 engenharia + SMOTENC (robust)"]
COR = {c: GRAY if c.startswith("C0") else BLUE if c.startswith("C1") else ORANGE for c in ORDEM}

## 1. Escalonamento sem SMOTENC: efeito nulo por construção (C2)

MinMax, Standard e Robust são transformações lineares crescentes. Elas preservam a **ordem** dos valores, logo os
quantis e as faixas da discretização caem nos mesmos registros. Para a Rede Bayesiana discreta, os dados escalonados
são **indistinguíveis** dos originais. A célula abaixo verifica isso em um fold real, comparando as probabilidades
previstas.

In [ ]:
df = pd.read_parquet(RAIZ / "data" / "processed" / "features.parquet")
raw_aux = load_raw()
assert df.appointment_date.reset_index(drop=True).equals(raw_aux.appointment_date.reset_index(drop=True))
assert np.array_equal(df.no_show, raw_aux.no_show)
df["city"] = raw_aux.city.to_numpy()
raw_validation = pd.read_csv(RAW_PATH)
validation_groups = patient_groups(df.patient_id, records=raw_validation)
y = df.no_show.to_numpy()
X = df[CATEGORICAL + CONTINUOUS + ["city"]]
_, _, tr, te = next(repeated_group_folds(y, validation_groups, n_splits=5, n_repeats=1))

criterio_cv = protocolo["settings"]["cv_scoring"]
p_base, mb_base, _ = fit_predict(X.iloc[tr], y[tr], X.iloc[te], CATEGORICAL, CONTINUOUS, scoring=criterio_cv)
print(f"C1 sem escalonador → Manto: {mb_base}")
for nome, Scaler in SCALERS.items():
    sc = Scaler().fit(X.iloc[tr][CONTINUOUS])                 # sklearn ignora NaN no ajuste e os preserva
    Xs = X.copy()
    Xs[CONTINUOUS] = sc.transform(X[CONTINUOUS])
    p, mb, _ = fit_predict(Xs.iloc[tr], y[tr], Xs.iloc[te], CATEGORICAL, CONTINUOUS, scoring=criterio_cv)
    print(f"C2 {nome:8s} → Manto: {mb} | probabilidades idênticas a C1: {np.allclose(p, p_base)}")

Confirmado: **o escalonamento sozinho não altera nada na Rede Bayesiana discreta**. O único canal pelo qual o
escalonador pode influenciar o modelo é a **distância usada pelo SMOTENC** para escolher vizinhos (C3). Por isso,
C2 não foi repetida nos 15 folds.

## 2. Validação cruzada: resultados (H3)

In [ ]:
resumo = CV.groupby("config").agg(
    roc_auc=("roc_auc", "mean"), roc_dp=("roc_auc", "std"), pr_auc=("pr_auc", "mean"), pr_dp=("pr_auc", "std"),
    brier=("brier", "mean"), p_media=("p_media", "mean"), n_params=("n_params", "mean"),
).loc[ORDEM]
resumo["Manto mais frequente"] = CV.groupby("config").manto.agg(lambda s: s.value_counts().index[0]).loc[ORDEM]
resumo.style.format({"roc_auc": "{:.4f}", "roc_dp": "±{:.4f}", "pr_auc": "{:.4f}", "pr_dp": "±{:.4f}",
                     "brier": "{:.4f}", "p_media": "{:.1%}", "n_params": "{:,.0f}"}) \
    .highlight_max(subset=["roc_auc", "pr_auc"], color="#d6e6f8").highlight_min(subset=["brier"], color="#d6e6f8")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
ypos = {c: i for i, c in enumerate(ORDEM[::-1])}
rng = np.random.default_rng(0)
for ax, (met, titulo) in zip(axes, [("pr_auc", "PR-AUC (↑)"), ("roc_auc", "ROC-AUC (↑)"), ("brier", "Brier (↓)")]):
    for c in ORDEM:
        v = CV.loc[CV.config == c, met]
        ax.scatter(v, ypos[c] + rng.uniform(-0.15, 0.15, len(v)), s=14, color=COR[c], alpha=0.6, linewidths=0)
        ax.plot(v.mean(), ypos[c], "|", color=INK, ms=16, mew=2)
    ax.set_title(titulo); ax.grid(axis="y", visible=False)
axes[0].set_yticks(list(ypos.values()), list(ypos.keys()), fontsize=8)
fig.suptitle("Validação cruzada agrupada, 15 folds (pontos = folds, traço = média)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Wilcoxon pareado com correção de Holm (exploratório).** A correção considera as comparações listadas, mas não elimina a dependência dos folds repetidos nem a exploração anterior desta base. Os p-valores não constituem confirmação estatística independente de H3; priorizar magnitude, dispersão, pares favoráveis e comportamento temporal.


In [ ]:
def holm(p):
    p = np.asarray(p); ordem = np.argsort(p); m = len(p)
    ajust = np.maximum.accumulate((m - np.arange(m)) * p[ordem]).clip(max=1)
    out = np.empty(m); out[ordem] = ajust
    return out

COMPARACOES = [(c, "C0 bruto") for c in ORDEM[1:]] + [(c, "C1 engenharia") for c in ORDEM[3:]]
linhas = []
for met in ["roc_auc", "pr_auc"]:
    piv = CV.pivot_table(index=["rep", "fold"], columns="config", values=met)
    for a, b in COMPARACOES:
        linhas.append({"comparação": f"{a}  ×  {b}", "métrica": met, **paired_wilcoxon(piv[a], piv[b])})
testes = pd.DataFrame(linhas)
testes["p_holm"] = holm(testes.p)
testes["sinal exploratório"] = np.where(testes.p_holm >= 0.05, "inconclusivo",
                               np.where(testes.diferença_mediana > 0, "favorável", "desfavorável"))
testes.set_index(["comparação", "métrica"]).style.format(
    {"diferença_mediana": "{:+.4f}", "p": "{:.4f}", "p_holm": "{:.4f}"}) \
    .map(lambda v: f"color: {BLUE}; font-weight: bold" if v == "favorável" else
         f"color: {ORANGE}; font-weight: bold" if v == "desfavorável" else "", subset=["sinal exploratório"])

## 3. Diagnóstico de incoerências do SMOTENC sem reparo

O SMOTENC cria um registro sintético de no-show a partir de um registro real e de um vizinho: as contínuas são
**interpoladas** e as categóricas recebem a **moda dos vizinhos**. Em agendamentos, isso quebra duas estruturas:

1. **O clima é um atributo do dia**, igual para todos os agendamentos daquela data. Interpolar a chuva entre dois
   pacientes de dias diferentes cria dias que **nunca existiram**: chuva de 7 mm/h com categoria "sem chuva", por
   exemplo.
2. **Combinações impossíveis viram "assinatura" de registro sintético.** A chuva numérica é interpolada, mas a
   categoria de chuva vem da moda dos vizinhos, então as duas deixam de ser coerentes. Essas combinações **só existem
   nos sintéticos**, e como todos os sintéticos são no-show, elas têm 100% de no-show no treino. Isso pode levar a estrutura a reconhecer a fabricação dos registros. A presença de chuva no Manto é um indício, mas não isola causalmente esse mecanismo.

A célula abaixo mede o mecanismo **sem reparo** em um fold; não representa o braço corrigido da nova execução.

In [ ]:
Xtr = X.iloc[tr]
Xr, yr = smotenc_resample(Xtr, y[tr], CATEGORICAL, CONTINUOUS, scaler="robust")
sint = Xr.iloc[len(Xtr):]
print(f"Treino original: {len(Xtr):,} ({y[tr].mean():.1%} no-show) → reamostrado: {len(Xr):,} ({yr.mean():.0%}); "
      f"{len(sint):,} sintéticos")

# 1. combinações de clima que não existem em nenhum dia real
dias_reais = set(map(tuple, df[["max_temp_day", "max_rain_day"]].dropna().round(2).to_numpy()))
clima_sint = sint[["max_temp_day", "max_rain_day"]].dropna().round(2)
inexistentes = np.mean([tuple(r) not in dias_reais for r in clima_sint.to_numpy()])
print(f"Sintéticos com (temperatura, chuva) que não existem em nenhum dia real: {inexistentes:.0%}")

from noshow.weather import rain_category
incoerente = (rain_category(sint.max_rain_day) != sint.rain_intensity) & sint.max_rain_day.notna()
print(f"Sintéticos com rain_intensity incoerente com max_rain_day: {incoerente.mean():.0%} "
      f"(nos dados reais: {((rain_category(Xtr.max_rain_day) != Xtr.rain_intensity) & Xtr.max_rain_day.notna()).mean():.0%})")

In [ ]:
from noshow.features import Discretizer
sintetico = np.r_[np.zeros(len(Xtr), bool), np.ones(len(Xr) - len(Xtr), bool)]
D = Discretizer(columns=CONTINUOUS).fit(Xr).transform(Xr)
combo = D.max_rain_day + "  |  " + D.rain_intensity
impossivel = ~combo.isin(set(combo[~sintetico]))
print(f"Sintéticos com combinação (faixa de max_rain_day, rain_intensity) que não existe nos dados reais: "
      f"{impossivel[sintetico].mean():.0%}")
print(f"Taxa de no-show no treino reamostrado → combinação real: {yr[~impossivel].mean():.1%} | "
      f"combinação só-sintética: {yr[impossivel].mean():.1%} (n = {impossivel.sum():,})")
tab = pd.crosstab(combo[impossivel], yr[impossivel]).rename(columns={1: "no-show (todos sintéticos)"})
tab.sort_values(tab.columns[-1], ascending=False).head(6)

In [ ]:
freq = (CV.assign(chuva_no_manto=CV.manto.str.contains("rain"))
          .groupby("config").chuva_no_manto.mean().loc[ORDEM])
print("Fração dos folds em que alguma variável de chuva entra no Manto de Markov:")
print(freq.map("{:.0%}".format).to_string())

## 4. H1: PR-AUC e Recall

A hipótese exige melhora em PR-AUC e Recall. A PR-AUC resume a ordenação; para comparar recall com o mesmo esforço, cada modelo recebe orçamento de 10% dos agendamentos. Empates no ponto de corte usam o número esperado de acertos sob sorteio uniforme, sem favorecer a ordem das linhas. Os limiares 0,5 e prevalência do treino são mantidos como análises descritivas. As probabilidades do protocolo atual já incluem correção após SMOTENC.


In [ ]:
budget_cols = ["recall@top10", "precision@top10", "pct_alertas@top10"]
CV.groupby("config")[budget_cols].agg(["mean", "std"]).reindex(ORDEM).style.format("{:.4f}")


In [ ]:
cols = ["accuracy", "recall", "precision", "f1", "balanced_accuracy", "pct_alertas"]
lim = pd.concat({
    "limiar 0,5": CV.groupby("config")[[f"{c}@050" for c in cols]].mean().set_axis(cols, axis=1),
    "limiar = prevalência": CV.groupby("config")[[f"{c}@prev" for c in cols]].mean().set_axis(cols, axis=1),
}, axis=1).loc[ORDEM]
lim.style.format("{:.3f}").format("{:.1%}", subset=[(l, "pct_alertas") for l in ["limiar 0,5", "limiar = prevalência"]])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for c in ORDEM:
    for sufixo, marcador in [("050", "o"), ("prev", "s")]:
        sub = CV[CV.config == c]
        ax.scatter(sub[f"pct_alertas@{sufixo}"].mean(), sub[f"recall@{sufixo}"].mean(), color=COR[c], marker=marcador,
                   s=70, edgecolors="white", linewidths=1.5, zorder=3)
x = np.linspace(0, 1, 50)
ax.plot(x, x, color=GRAY, ls="--", lw=1)
ax.text(0.62, 0.55, "alertar ao acaso\n(recall = % alertado)", fontsize=8, color=INK)
ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel("% dos agendamentos marcados como risco"); ax.set_ylabel("Recall (faltas detectadas)")
ax.set_title("Recall × volume de alertas")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=GRAY, label="C0 bruto"),
                   Line2D([], [], marker="o", ls="", color=BLUE, label="C1 engenharia"),
                   Line2D([], [], marker="o", ls="", color=ORANGE, label="C3 + SMOTENC"),
                   Line2D([], [], marker="o", ls="", color=INK, mfc="white", label="limiar 0,5"),
                   Line2D([], [], marker="s", ls="", color=INK, mfc="white", label="limiar = prevalência")],
          frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

A comparação de recall deve considerar o volume de alertas. A tabela com orçamento fixo de 10% evita atribuir vantagem a um modelo apenas porque ele alerta mais. A PR-AUC resume a relação entre precisão e recall ao longo dos limiares.


## 5. Validação temporal: o modelo prevê o futuro?

Treino com 2016–2020 e teste com 2021–2022 (fora: ano e período). É o cenário mais próximo de produção. Aqui os
dois critérios de estrutura são comparados: o AIC teve maior PR-AUC na validação cruzada agrupada de H2 (notebook 03), mas um
critério mais permissivo pode estar aprendendo padrões de calendário que não se repetem no futuro.

In [ ]:
tab = TEMP.set_index(["criterio", "config"])[["roc_auc", "pr_auc", "brier", "p_media", "n_params", "manto"]]
tab = tab.reindex(pd.MultiIndex.from_product([["AIC", "BIC"], ORDEM]))
tab.style.format({"roc_auc": "{:.4f}", "pr_auc": "{:.4f}", "brier": "{:.4f}", "p_media": "{:.1%}", "n_params": "{:,.0f}"}) \
    .highlight_max(subset=["roc_auc", "pr_auc"], color="#d6e6f8").highlight_min(subset=["brier"], color="#d6e6f8")

In [ ]:
from sklearn.metrics import roc_auc_score
ano = df.appointment_date.dt.year
teste = df[ano >= 2021]
ok = teste.taxa_faltas_anteriores.notna()
print(f"Registros com ID reconstruído (histórico disponível): treino ≤2020 {df[ano <= 2020].patient_id.notna().mean():.0%} "
      f"→ teste 2021–2022 {teste.patient_id.notna().mean():.0%}")
print(f"ROC-AUC de taxa_faltas_anteriores sozinha no teste, só onde existe histórico: "
      f"{roc_auc_score(teste.no_show[ok], teste.taxa_faltas_anteriores[ok]):.3f}")

**Leitura da validação temporal.** Comparar controle e engenharia com o mesmo critério de estrutura. O BIC é principal; AIC é sensibilidade. Verificar se ganhos na CV permanecem em 2021–2022, levando em conta deriva de prevalência e redução da cobertura de cadastro. O histórico usa apenas dias anteriores à consulta, em regime sequencial. Clima observado do próprio dia limita a interpretação operacional.


In [ ]:
prev_tr = df[df.appointment_date.dt.year <= 2020].no_show.mean()
prev_te = df[df.appointment_date.dt.year >= 2021].no_show.mean()
print(f"Prevalência de no-show: treino (≤2020) {prev_tr:.1%} → teste (2021–2022) {prev_te:.1%}")
print("Compare a probabilidade média de cada modelo com a prevalência do teste para verificar subestimação ou superestimação do risco.")

## 6. Síntese

### Interpretação após corrigir a sobreposição

A auditoria verificou zero registros equivalentes entre treino e teste nos 15 folds, e entre treino interno e validação nos 15 holdouts. Todas as 49.593 linhas foram preservadas. A assinatura inclui os campos originais, data e horário; exclui alvo e motivo. Pessoas sem ID em dias diferentes continuam sem separação garantida.

**H1, valores descritivos:** engenharia sem SMOTENC obteve PR-AUC média 0,1426 e recall@10% 18,44%; o bruto obteve 0,1324 e 15,68%. O melhor braço pela AP média corrigida foi **bruto + SMOTENC Robust**, com PR-AUC 0,1762, ROC-AUC 0,6665 e recall@10% 20,78%.

**H3:** 0 dos quatro C3 superam o bruto simultaneamente nas médias de PR-AUC e ROC-AUC. Os C3 têm PR-AUC entre 0,1322 e 0,1405, e ROC-AUC entre 0,5934 e 0,6084. O controle bruto tem ROC-AUC 0,6165. Ganhos em um braço não demonstram superioridade geral do framework; os testes sobre folds repetidos são exploratórios.

**Impacto da correção das divisões:** a AP de bruto + SMOTENC Robust passou de 0,1760 para 0,1762. Essa comparação de médias é descritiva: os folds e a seleção interna mudaram, portanto o fold antigo e o novo não são pares com os mesmos registros.

**Comparação dos tratamentos com os grupos já corrigidos:** bruto + SMOTENC passa de 0,1561 para 0,1762 ao aplicar o perfil ajustado, com 14/15 folds favoráveis. Aqui, os dois perfis usam exatamente os mesmos folds corrigidos; “anterior” significa o pré-processamento anterior, não a divisão com sobreposição.

**Temporal:** as 28 avaliações BIC/AIC dos dois perfis reproduziram as métricas anteriores, pois a separação por data e o holdout temporal permanecem os mesmos. Clima observado e histórico sequencial ainda exigem definir o instante da previsão. O período já foi explorado no desenvolvimento.

A auditoria das divisões está em `resultados/artigo_sem_sobreposicao/auditoria_divisoes.csv`.


## 7. Comparação pareada com a versão anterior

A versão anterior já contém isolamento de categorias, ajuste de ESS, reparo de idade/clima e correção da proporção de classes. A nova acrescenta ajuste de faixas e de balanceamento e a relação exata da taxa histórica sintética. As duas versões usam os mesmos treinos, testes e critérios. A comparação abaixo não usa os resultados históricos com outro critério de estrutura.


In [ ]:
comparison_path = RAIZ / "resultados" / "artigo_sem_sobreposicao" / "comparacao" / "comparacao_cv.csv"
if comparison_path.exists():
    comparacao = pd.read_csv(comparison_path)
    display(comparacao[comparacao.metrica.isin(["pr_auc", "recall@top10", "brier"])].set_index(["codigo", "metrica"]))
else:
    print("A comparação entre versões é opcional. Para calculá-la, execute scripts/compare_article.py conforme o README.")
